In [1]:
import os
import re
import time
import shutil
import pyodbc
import fnmatch
import numpy as np
import pandas as pd
import win32com.client
from math import floor
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

In [2]:
# Generate random iri
def generate_parts(target_values, num_parts, tolerance):
    parts_list = []
    for target_value in target_values:
        total_sum = target_value * num_parts
        while True:
            # Generate random parts
            parts = np.random.uniform(low=total_sum / num_parts * 0.9, high=total_sum / num_parts * 1.1, size=num_parts)
            # Ensure the sum is correct
            if np.abs(np.sum(parts) - total_sum) < tolerance:
                parts_list.append(parts)
                break
            
    return parts_list

# Find all relevant CSV files and process them
def process_csv_files(path):
    all_iri_dataframes = [] # empty list
    all_rutting_dataframes = [] # empty list
    
    for root, dirs, files in os.walk(path):
        # Find files
        iri_files = [f for f in files if f.endswith('.csv') and 'xw_iri_qgis' in f]
        rutting_files = [f for f in files if f.endswith('.csv') and 'xw_rutting' in f]

        # Process 'xw_iri_qgis' files
        for filename in iri_files:
            file_path = os.path.join(root, filename)
            iri_df = pd.read_csv(file_path, delimiter=';')
            iri_df.columns = iri_df.columns.str.strip()
            survey_code = filename.split('_')[3].split('.')[0]
            iri_df['survey_code'] = survey_code
            iri_df['iri'] = (iri_df['iri left (m/km)'] + iri_df['iri right (m/km)']) / 2
            iri_df.drop(columns=['geometry'], errors='ignore', inplace=True)

            # Generate random values for iri_lane
            target_values = iri_df['iri']
            num_parts = 4
            tolerance = 0.3
            parts_list = generate_parts(target_values, num_parts, tolerance)

            # Expand DataFrame by repeating the rows
            iri_df = iri_df.loc[iri_df.index.repeat(num_parts)].reset_index(drop=True)
            iri_df['iri_lane'] = np.concatenate(parts_list)

            increment = 5 if fnmatch.fnmatch(filename, '*xw_iri_qgis*') else 5
            iri_df['event_start'] = range(0, len(iri_df) * increment, increment)
            iri_df['event_end'] = iri_df['event_start'] + increment

            # Append the processed IRI DataFrame to the list
            all_iri_dataframes.append(iri_df)

        # Process 'xw_rutting' files
        for filename in rutting_files:
            file_path = os.path.join(root, filename)
            rut_df = pd.read_csv(file_path, delimiter=';')
            rut_df.columns = rut_df.columns.str.strip()
            if 'Unnamed: 5' in rut_df.columns:
                rut_df.drop(columns=['Unnamed: 5'], inplace=True, errors='ignore')
            else:
                pass
            increment = 5 if fnmatch.fnmatch(filename, '*xw_rutting*') else 5
            rut_df['event_start'] = range(0, len(rut_df) * increment, increment)
            rut_df['event_end'] = rut_df['event_start'] + increment
            rut_df['rut_chainage'] = rut_df['event_start']
            survey_code = filename.split('_')[2].split('.')[0]
            rut_df['survey_code'] = survey_code
            if 'qgis_shape' in rut_df.columns:
                rut_df['rut_point_x'] = rut_df['qgis_shape'].apply(lambda x: float(x.split('(')[1].split(')')[0].split(',')[0].split(' ')[1]))
                rut_df['rut_point_y'] = rut_df['qgis_shape'].apply(lambda x: float(x.split('(')[1].split(')')[0].split(',')[0].split(' ')[0]))
                
                # Apply interpolation with a limit to avoid interpolating across large gaps
                rut_df['rut_point_x'] = rut_df['rut_point_x'].interpolate(method='linear', limit_direction='both')
                rut_df['rut_point_y'] = rut_df['rut_point_y'].interpolate(method='linear', limit_direction='both')

                # Forward/backward fill to close gaps
                rut_df['rut_point_x'].fillna(method='ffill', inplace=True)
                rut_df['rut_point_x'].fillna(method='bfill', inplace=True)
                rut_df['rut_point_y'].fillna(method='ffill', inplace=True)
                rut_df['rut_point_y'].fillna(method='bfill', inplace=True)

                # Replace remaining NaN values with 0 (optional)
                rut_df['rut_point_x'].fillna(0, inplace=True)
                rut_df['rut_point_y'].fillna(0, inplace=True)
                
                rut_df.drop(columns=['qgis_shape'], inplace=True)
            else:
                print("not found 'qgis_shape' or it have a white space ! ")
        
            rut_df.rename(columns={'#Date':'Date', 'left rutting height': 'left_rutting', 'right rutting height': 'right_rutting', 'average height': 'avg_rutting'}, inplace=True)

            all_rutting_dataframes.append(rut_df)

    if all_iri_dataframes:
        iri_dataframes = pd.concat(all_iri_dataframes, ignore_index=True)
    else:
        iri_dataframes = pd.DataFrame()

    if all_rutting_dataframes:
        rutting_dataframes = pd.concat(all_rutting_dataframes, ignore_index=True)
    else:
        rutting_dataframes = pd.DataFrame()
        
    print(f"✅ Finished processing: .CSV files.")
    return iri_dataframes, rutting_dataframes

In [3]:
# Perform the left join on xw_rutting and xw_iri_qgis
def left_join_dataframes(df_rutting, df_iri):
    df_merged = pd.merge(df_rutting, df_iri, how='left', on=['event_start', 'event_end', 'survey_code'], suffixes=('_rutting', '_iri'))
    
    # print("Merged DataFrame columns:", df_merged.columns)

    result = df_merged[
        (df_merged['rut_chainage'] >= df_merged['event_start']) &
        (df_merged['rut_chainage'] < df_merged['event_end'])
    ]
    return result

# Perform jpg file and frame number
def get_jpg_filenames(directory):
    jpg_dict = {}
    for root, dirs, files in os.walk(directory):
        jpg_files = [f for f in files if f.endswith('.jpg')]
        if jpg_files:
            folder_name = os.path.basename(os.path.dirname(root))
            jpg_dict[folder_name] = len(jpg_files)
            
    frame_df = pd.DataFrame(list(jpg_dict.items()), columns=['survey_code','frame_num'])
    frame_df['survey_code'] = frame_df['survey_code'].str.replace(
        r'_(\d+)', lambda m: f"RUN{int(m.group(1)):02d}", regex=True
    )
    
    detailed_df = pd.DataFrame(columns=['frame_num', 'survey_code'])
    for index, row in frame_df.iterrows():
        pic_counts = range(1, int(row['frame_num']) + 1)
        temp_df = pd.DataFrame({
            'frame_num': pic_counts,
            'survey_code': row['survey_code'],
        })
        detailed_df = pd.concat([detailed_df, temp_df], ignore_index=True)

    return detailed_df

def add_frame_num_to_joined_df(joined_df, derived_values, frame_numbers):
    joined_df['frame_num_ch'] = pd.NA
    joined_df['frame_num'] = pd.NA
    
    derived_to_frame_mapping = pd.DataFrame({
        'frame_num_ch': derived_values,
        'frame_num': frame_numbers
    })
    
    for i, frame_num_ch in enumerate(derived_values):
        mask = (joined_df['event_start'] <= frame_num_ch) & (joined_df['event_end'] >= frame_num_ch)
        joined_df.loc[mask, 'frame_num_ch'] = frame_num_ch
        joined_df.loc[mask, 'frame_num'] = frame_numbers[i]
        
    return joined_df

def process_fainal_df(output_dir):
    frame_numbers_df = get_jpg_filenames(output_dir)  # This returns a DataFrame
    frame_numbers = frame_numbers_df['frame_num'].astype(int).tolist()
    print(frame_numbers)
    iri_dataframes, rutting_dataframes = process_csv_files(output_dir)

    joined_df = left_join_dataframes(rutting_dataframes, iri_dataframes)

    grouped_df = joined_df.groupby('survey_code').agg(
        max_chainage=('rut_chainage', 'max'),
        min_chainage=('rut_chainage', 'min')
    ).reset_index()

    joined_df = pd.merge(joined_df, grouped_df, on='survey_code', how='left')
    
    max_event_start = joined_df['event_start'].max()

    # Calculate derived values
    derived_values = [floor((max_event_start * num) / max(frame_numbers)) for num in frame_numbers]
    # derived_values = [((max_event_start * num) / max(frame_numbers)) for num in frame_numbers]

    # Add frame numbers to the joined DataFrame
    final_df = add_frame_num_to_joined_df(joined_df, derived_values, frame_numbers)
    
    final_df = final_df.rename(columns={'rut_chainage':'chainage'})
    
    selected_columns = [
        'left_rutting', 'right_rutting', 'avg_rutting', 'event_start', 'event_end', 'survey_code',
        'rut_point_x', 'rut_point_y', 'Date', 'iri left (m/km)', 'iri right (m/km)', 'iri', 'iri_lane', 
        'chainage', 'max_chainage', 'min_chainage', 'frame_num', 'frame_num_ch'
    ] 
    
    selected_columns = [col for col in selected_columns if col in final_df.columns]
    # final_df = final_df[final_df['iri'].notnull()][selected_columns]
    
    return joined_df, derived_values, final_df

In [5]:
output_dir = r'D:\xenomatix\output'
joined_df, derived_values, final_df = process_fainal_df(output_dir)

# final_df.to_csv('joinjoin.csv')
joined_df.tail(15)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 22

,Date_rutting,left_rutting,right_rutting,avg_rutting,event_start,event_end,rut_chainage,survey_code,rut_point_x,rut_point_y,Date_iri,iri left (m/km),iri Std left (m/km),iri right (m/km),iri Std right (m/km),worst iri (m/km),iri difference (m/km),iri,iri_lane,max_chainage,min_chainage,frame_num_ch,frame_num
2430,26/07/2024,7.82,0.00,3.91,5965,5970,5965,20240726RUN03,13.849291,100.444227,26/07/2024,8.031086,4.547960,10.226871,5.272060,10.226871,2.195785,9.128979,8.478665,6035,0,5969,850
2431,26/07/2024,4.07,0.00,2.04,5970,5975,5970,20240726RUN03,13.849312,100.444186,26/07/2024,8.031086,4.547960,10.226871,5.272060,10.226871,2.195785,9.128979,9.988178,6035,0,<NA>,<NA>
2432,26/07/2024,1.07,0.03,0.55,5975,5980,5975,20240726RUN03,13.849332,100.444145,26/07/2024,8.031086,4.547960,10.226871,5.272060,10.226871,2.195785,9.128979,9.762933,6035,0,5976,851
2433,26/07/2024,0.26,0.08,0.17,5980,5985,5980,20240726RUN03,13.849353,100.444104,26/07/2024,8.690157,8.111755,10.062783,8.875238,10.062783,1.372626,9.376470,8.643300,6035,0,5983,852
2434,26/07/2024,0.30,0.45,0.38,5985,5990,5985,20240726RUN03,13.849374,100.444063,26/07/2024,8.690157,8.111755,10.062783,8.875238,10.062783,1.372626,9.376470,9.410157,6035,0,5990,853
2435,26/07/2024,0.26,2.96,1.61,5990,5995,5990,20240726RUN03,13.849395,100.444022,26/07/2024,8.690157,8.111755,10.062783,8.875238,10.062783,1.372626,9.376470,9.524252,6035,0,5990,853
2436,26/07/2024,0.41,6.50,3.46,5995,6000,5995,20240726RUN03,13.849415,100.443981,26/07/2024,8.690157,8.111755,10.062783,8.875238,10.062783,1.372626,9.376470,10.151578,6035,0,5997,854
2437,26/07/2024,0.34,6.09,3.22,6000,6005,6000,20240726RUN03,13.849436,100.443940,26/07/2024,15.840459,12.255976,16.280330,12.531867,16.280330,0.439871,16.060394,16.147026,6035,0,6004,855
2438,26/07/2024,5.66,2.46,4.06,6005,6010,6005,20240726RUN03,13.849459,100.443900,26/07/2024,15.840459,12.255976,16.280330,12.531867,16.280330,0.439871,16.060394,16.867172,6035,0,<NA>,<NA>
2439,26/07/2024,6.84,0.27,3.56,6010,6015,6010,20240726RUN03,13.849488,100.443864,26/07/2024,15.840459,12.255976,16.280330,12.531867,16.280330,0.439871,16.060394,14.597924,6035,0,6011,856


In [7]:
df = pd.DataFrame(derived_values)

df.head(1)

,0
0,7


In [8]:
final_df.head(10)

,Date_rutting,left_rutting,right_rutting,avg_rutting,event_start,event_end,chainage,survey_code,rut_point_x,rut_point_y,Date_iri,iri left (m/km),iri Std left (m/km),iri right (m/km),iri Std right (m/km),worst iri (m/km),iri difference (m/km),iri,iri_lane,max_chainage,min_chainage,frame_num_ch,frame_num
0,26/07/2024,2.08,0.03,1.06,0,5,0,20240726RUN02,13.853040,100.443802,26/07/2024,3.865407,2.804227,3.482863,2.631087,3.865407,0.382544,3.674135,3.663915,6180,0,<NA>,<NA>
1,26/07/2024,1.21,0.00,0.60,5,10,5,20240726RUN02,13.852996,100.443810,26/07/2024,3.865407,2.804227,3.482863,2.631087,3.865407,0.382544,3.674135,3.480324,6180,0,7,1
2,26/07/2024,1.95,0.00,0.98,10,15,10,20240726RUN02,13.852951,100.443819,26/07/2024,3.865407,2.804227,3.482863,2.631087,3.865407,0.382544,3.674135,3.662004,6180,0,14,2
3,26/07/2024,1.69,0.00,0.84,15,20,15,20240726RUN02,13.852907,100.443827,26/07/2024,3.865407,2.804227,3.482863,2.631087,3.865407,0.382544,3.674135,3.732081,6180,0,<NA>,<NA>
4,26/07/2024,2.73,0.04,1.38,20,25,20,20240726RUN02,13.852862,100.443834,26/07/2024,3.547611,2.251651,3.629724,2.853411,3.629724,0.082113,3.588667,3.672796,6180,0,21,3
5,26/07/2024,2.68,0.09,1.38,25,30,25,20240726RUN02,13.852818,100.443841,26/07/2024,3.547611,2.251651,3.629724,2.853411,3.629724,0.082113,3.588667,3.664614,6180,0,28,4
6,26/07/2024,2.29,0.00,1.14,30,35,30,20240726RUN02,13.852773,100.443846,26/07/2024,3.547611,2.251651,3.629724,2.853411,3.629724,0.082113,3.588667,3.623513,6180,0,35,5
7,26/07/2024,2.38,0.00,1.19,35,40,35,20240726RUN02,13.852728,100.443852,26/07/2024,3.547611,2.251651,3.629724,2.853411,3.629724,0.082113,3.588667,3.351260,6180,0,35,5
8,26/07/2024,1.50,0.00,0.75,40,45,40,20240726RUN02,13.852683,100.443856,26/07/2024,3.097305,2.065051,3.022079,2.184983,3.097305,0.075226,3.059692,3.276841,6180,0,42,6
9,26/07/2024,3.24,0.02,1.63,45,50,45,20240726RUN02,13.852638,100.443860,26/07/2024,3.097305,2.065051,3.022079,2.184983,3.097305,0.075226,3.059692,2.787513,6180,0,49,7


In [20]:
def find_csv_files(start_dir, prefix='log_'):
    csv_files = []
    for dirpath, dirnames, filenames in os.walk(start_dir):
        for filename in fnmatch.filter(filenames, f'{prefix}*.xlsx'):
            csv_files.append(os.path.join(dirpath, filename))
    return csv_files

def main(final_df, output_dir):
    for survey_date in os.listdir(output_dir): # eg. base_dir = r"D:\xenomatixs"
        path = os.path.join(output_dir, survey_date, 'Output')
        mdb = os.path.join(output_dir, survey_date, 'Data')
        
        log_csv_files = find_csv_files(path)
        if log_csv_files:
            log_df = pd.read_excel(log_csv_files[0])
            log_df.rename(columns={'ผิว': 'event_name', 'link_id ระบบ': 'section_id'}, inplace=True)
            log_df.columns = log_df.columns.str.strip()

            folder_names = [name for name in os.listdir(path) if os.path.isdir(os.path.join(path, name))]
            for folder_name in folder_names:
                print(f"🔄 Processing folder: {folder_name}")
                
                # Perform the initial merge and filter rows where frame_num is between numb_start and numb_end
                merged_df = pd.merge(final_df, log_df, how='left', on=['survey_code'], suffixes=('_final_df', '_log_df'))
                merged_df = merged_df[(merged_df['frame_num'] >= merged_df['numb_start']) & 
                                    (merged_df['frame_num'] <= merged_df['numb_end'])]
                
                filtered_df = merged_df[merged_df['survey_code'] == folder_name]
                run_code = re.sub(r'RUN0*(\d+)', r'_\1', folder_name)
                
                # add filter_df as min_chainage and max_chainage is group by numb_start and numb_end and merge to merged_df
                filter_df = merged_df.groupby(['numb_start', 'numb_end'], group_keys=False).agg(
                    min_chainage=('chainage', 'min'),
                    max_chainage=('chainage', 'max')
                ).reset_index()
                
                merged_df = pd.merge(merged_df, filter_df, on=['numb_start', 'numb_end'], how='left')
                filtered_df = pd.merge(filtered_df, filter_df, on=['numb_start', 'numb_end'], how='left')
                
    return merged_df, filtered_df

In [17]:
merged_df, filtered_df = main(final_df, output_dir)

🔄 Processing folder: 20240726RUN02
🔄 Processing folder: 20240726RUN03


In [23]:
merged_df.tail(20)

,Date_rutting,left_rutting,right_rutting,avg_rutting,event_start,event_end,chainage,survey_code,rut_point_x,rut_point_y,Date_iri,iri left (m/km),iri Std left (m/km),iri right (m/km),iri Std right (m/km),worst iri (m/km),iri difference (m/km),iri,iri_lane,max_chainage,min_chainage,frame_num_ch,frame_num,linkid,ramp_id,section_id,numb_start,numb_end,km_start,km_end,length,length_KM,lane,event_name,date,route,remark
4823,26/07/2024,1.41,0.00,0.70,5870,5875,5870,20240726RUN03,13.848905,100.445012,26/07/2024,4.546525,3.625785,3.041138,2.332667,4.546525,1.505386,3.793832,3.866734,6035,0,5871,836,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN
4825,26/07/2024,1.12,0.10,0.61,5875,5880,5875,20240726RUN03,13.848925,100.444971,26/07/2024,4.546525,3.625785,3.041138,2.332667,4.546525,1.505386,3.793832,3.959766,6035,0,5878,837,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN
4827,26/07/2024,1.66,0.00,0.83,5880,5885,5880,20240726RUN03,13.848946,100.444930,26/07/2024,3.823193,2.212825,4.534176,2.474581,4.534176,0.710983,4.178685,3.947023,6035,0,5885,838,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN
4829,26/07/2024,1.27,0.35,0.81,5885,5890,5885,20240726RUN03,13.848966,100.444889,26/07/2024,3.823193,2.212825,4.534176,2.474581,4.534176,0.710983,4.178685,4.245603,6035,0,5885,838,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN
4831,26/07/2024,0.85,0.00,0.42,5890,5895,5890,20240726RUN03,13.848987,100.444847,26/07/2024,3.823193,2.212825,4.534176,2.474581,4.534176,0.710983,4.178685,4.131274,6035,0,5892,839,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN
4833,26/07/2024,0.52,0.10,0.31,5895,5900,5895,20240726RUN03,13.849007,100.444806,26/07/2024,3.823193,2.212825,4.534176,2.474581,4.534176,0.710983,4.178685,4.499075,6035,0,5899,840,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN
4837,26/07/2024,1.50,0.55,1.02,5905,5910,5905,20240726RUN03,13.849048,100.444724,26/07/2024,4.372934,2.932256,4.463496,2.969037,4.463496,0.090561,4.418215,4.647007,6035,0,5906,841,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN
4839,26/07/2024,1.04,0.17,0.60,5910,5915,5910,20240726RUN03,13.849069,100.444682,26/07/2024,4.372934,2.932256,4.463496,2.969037,4.463496,0.090561,4.418215,4.247576,6035,0,5913,842,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN
4841,26/07/2024,1.74,0.42,1.08,5915,5920,5915,20240726RUN03,13.849089,100.444641,26/07/2024,4.372934,2.932256,4.463496,2.969037,4.463496,0.090561,4.418215,4.793958,6035,0,5920,843,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN
4843,26/07/2024,3.31,0.17,1.74,5920,5925,5920,20240726RUN03,13.849110,100.444600,26/07/2024,3.018241,1.833108,4.021036,2.372752,4.021036,1.002795,3.519639,3.794200,6035,0,5920,843,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN


In [24]:
filtered_df.tail(20)

,Date_rutting,left_rutting,right_rutting,avg_rutting,event_start,event_end,chainage,survey_code,rut_point_x,rut_point_y,Date_iri,iri left (m/km),iri Std left (m/km),iri right (m/km),iri Std right (m/km),worst iri (m/km),iri difference (m/km),iri,iri_lane,max_chainage_x,min_chainage_x,frame_num_ch,frame_num,linkid,ramp_id,section_id,numb_start,numb_end,km_start,km_end,length,length_KM,lane,event_name,date,route,remark,min_chainage_y,max_chainage_y
948,26/07/2024,1.41,0.00,0.70,5870,5875,5870,20240726RUN03,13.848905,100.445012,26/07/2024,4.546525,3.625785,3.041138,2.332667,4.546525,1.505386,3.793832,3.866734,6035,0,5871,836,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN,1765,5980
949,26/07/2024,1.12,0.10,0.61,5875,5880,5875,20240726RUN03,13.848925,100.444971,26/07/2024,4.546525,3.625785,3.041138,2.332667,4.546525,1.505386,3.793832,3.959766,6035,0,5878,837,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN,1765,5980
950,26/07/2024,1.66,0.00,0.83,5880,5885,5880,20240726RUN03,13.848946,100.444930,26/07/2024,3.823193,2.212825,4.534176,2.474581,4.534176,0.710983,4.178685,3.947023,6035,0,5885,838,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN,1765,5980
951,26/07/2024,1.27,0.35,0.81,5885,5890,5885,20240726RUN03,13.848966,100.444889,26/07/2024,3.823193,2.212825,4.534176,2.474581,4.534176,0.710983,4.178685,4.245603,6035,0,5885,838,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN,1765,5980
952,26/07/2024,0.85,0.00,0.42,5890,5895,5890,20240726RUN03,13.848987,100.444847,26/07/2024,3.823193,2.212825,4.534176,2.474581,4.534176,0.710983,4.178685,4.131274,6035,0,5892,839,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN,1765,5980
953,26/07/2024,0.52,0.10,0.31,5895,5900,5895,20240726RUN03,13.849007,100.444806,26/07/2024,3.823193,2.212825,4.534176,2.474581,4.534176,0.710983,4.178685,4.499075,6035,0,5899,840,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN,1765,5980
954,26/07/2024,1.50,0.55,1.02,5905,5910,5905,20240726RUN03,13.849048,100.444724,26/07/2024,4.372934,2.932256,4.463496,2.969037,4.463496,0.090561,4.418215,4.647007,6035,0,5906,841,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN,1765,5980
955,26/07/2024,1.04,0.17,0.60,5910,5915,5910,20240726RUN03,13.849069,100.444682,26/07/2024,4.372934,2.932256,4.463496,2.969037,4.463496,0.090561,4.418215,4.247576,6035,0,5913,842,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN,1765,5980
956,26/07/2024,1.74,0.42,1.08,5915,5920,5915,20240726RUN03,13.849089,100.444641,26/07/2024,4.372934,2.932256,4.463496,2.969037,4.463496,0.090561,4.418215,4.793958,6035,0,5920,843,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN,1765,5980
957,26/07/2024,3.31,0.17,1.74,5920,5925,5920,20240726RUN03,13.849110,100.444600,26/07/2024,3.018241,1.833108,4.021036,2.372752,4.021036,1.002795,3.519639,3.794200,6035,0,5920,843,01900124046L1AC06,191,101655,252,852,0+000,4+300,4200,4.2,L2,AC,20240726,นบ.5038,NaN,1765,5980
